# Candle Backfill — 180-Day Gap Repair

This notebook connects to your MongoDB `candles` collection, discovers all existing
(connector, pair, interval) combos, shows gap statistics, and lets you select a
connector to backfill all missing candles for the last 180 days.

**Prerequisites:**
```bash
pip install pymongo requests pyyaml ipywidgets
```

In [1]:
# ── Cell 1: Configuration ────────────────────────────────────────────────────
import os

TRUENAS_IP      = os.environ.get("TRUENAS_LAN_IP", "192.168.1.54")
TRUENAS_DB_PASS = os.environ.get("MONGO_ROOT_PASSWORD", "mypass")
MONGO_URI = os.environ.get(
    "MONGO_URI",
    f"mongodb://admin:{TRUENAS_DB_PASS}@{TRUENAS_IP}:27017/quants_lab?authSource=admin",
)

MONGO_DATABASE = "quants_lab"
COLLECTION_NAME = "candles"

# How far back to look for gaps (days)
BACKFILL_DAYS = 180

# HTTP tuning
HTTP_TIMEOUT = 30
HTTP_MAX_RETRIES = 3
HTTP_RETRY_BACKOFF = 1.8

# Exchange base URLs
BASE_URLS = {
    "nonkyc": "https://api.nonkyc.io/api/v2",
    "binance": "https://api.binance.com",
    "mexc": "https://api.mexc.com",
    "coinbase": "https://api.exchange.coinbase.com",
}

# Per-exchange request delays (seconds)
REQUEST_DELAYS = {
    "nonkyc": 0.5,
    "binance": 0.12,
    "mexc": 0.2,
    "coinbase": 0.35,
}

# Max candles per API request
MAX_PER_REQUEST = {
    "nonkyc": 500,
    "binance": 1000,
    "mexc": 500,
    "coinbase": 300,
}

# Interval specifications
INTERVAL_SPEC = {
    "1m":  {"seconds": 60,     "binance": "1m",  "mexc": "1m",  "coinbase": 60,    "nonkyc": None},
    "3m":  {"seconds": 180,    "binance": "3m",  "mexc": None,  "coinbase": None,  "nonkyc": None},
    "5m":  {"seconds": 300,    "binance": "5m",  "mexc": "5m",  "coinbase": 300,   "nonkyc": 5},
    "15m": {"seconds": 900,    "binance": "15m", "mexc": "15m", "coinbase": 900,   "nonkyc": 15},
    "30m": {"seconds": 1800,   "binance": "30m", "mexc": "30m", "coinbase": None,  "nonkyc": 30},
    "1h":  {"seconds": 3600,   "binance": "1h",  "mexc": "60m", "coinbase": 3600,  "nonkyc": 60},
    "4h":  {"seconds": 14400,  "binance": "4h",  "mexc": "4h",  "coinbase": None,  "nonkyc": 240},
    "1d":  {"seconds": 86400,  "binance": "1d",  "mexc": "1d",  "coinbase": 86400, "nonkyc": 1440},
}

SCHEMA_VERSION = 3

print("Configuration loaded.")

Configuration loaded.


In [2]:
# ── Cell 2: Imports & Helpers ────────────────────────────────────────────────

import time
import math
from datetime import datetime, timezone
from typing import Any, Optional

import requests
from pymongo import MongoClient, UpdateOne
from pymongo.errors import BulkWriteError


def utc_now_ts() -> int:
    return int(time.time())


def align_floor(ts: int, step: int) -> int:
    return (ts // step) * step if step > 0 else ts


def last_closed_open_ts(now_ts: int, step: int) -> int:
    return align_floor(now_ts, step) - step


def fmt_ts(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")


def ts_to_iso8601(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat().replace("+00:00", "Z")


def safe_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return float("nan")


def to_hbot_pair(pair: str) -> str:
    p = pair.strip().upper()
    if "/" in p:
        base, quote = p.split("/", 1)
        return f"{base}-{quote}"
    if "_" in p:
        base, quote = p.split("_", 1)
        return f"{base}-{quote}"
    return p.replace("/", "-").replace("_", "-")


def split_hbot_pair(hbot_pair: str):
    return hbot_pair.split("-", 1)


def to_nonkyc_symbol(hbot_pair: str) -> str:
    return hbot_pair.replace("-", "_")


def to_binance_symbol(hbot_pair: str) -> str:
    return hbot_pair.replace("-", "")


def to_mexc_symbol(hbot_pair: str) -> str:
    return hbot_pair.replace("-", "")


def to_coinbase_product_id(hbot_pair: str) -> str:
    return hbot_pair  # already BASE-QUOTE


def compute_qc_flags(o, h, l, c, v):
    flags = []
    if h < max(o, c): flags.append("high_lt_open_close")
    if l > min(o, c): flags.append("low_gt_open_close")
    if h < l: flags.append("high_lt_low")
    if v < 0: flags.append("neg_volume")
    for name, val in [("open", o), ("high", h), ("low", l), ("close", c), ("volume", v)]:
        if val != val: flags.append(f"nan_{name}")
    return flags


print("Helpers loaded.")

Helpers loaded.


In [3]:
# ── Cell 3: HTTP + MongoDB helpers ───────────────────────────────────────────

session = requests.Session()


def http_get_json(url: str, params: dict = None, prefix: str = "") -> Any:
    """GET + JSON with retries."""
    last_err = None
    for attempt in range(1, HTTP_MAX_RETRIES + 1):
        try:
            resp = session.get(url, params=params, timeout=HTTP_TIMEOUT)
            if resp.status_code in (429, 418):
                retry_after = resp.headers.get("Retry-After")
                sleep_s = float(retry_after) if retry_after else HTTP_RETRY_BACKOFF ** attempt
                print(f"  {prefix}Rate limited (HTTP {resp.status_code}), sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue
            if resp.status_code in (500, 502, 503, 504):
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            last_err = e
            if attempt >= HTTP_MAX_RETRIES:
                break
            sleep_s = HTTP_RETRY_BACKOFF ** (attempt - 1)
            time.sleep(sleep_s)
    raise last_err


def normalize_candle_doc(
    connector, hbot_pair, interval, ts_open, open_, high, low, close,
    base_volume, interval_seconds, now_ts,
    quote_volume=None, trade_count=None,
    taker_buy_base_volume=None, taker_buy_quote_volume=None,
):
    base, quote = split_hbot_pair(hbot_pair)
    close_ts = ts_open + interval_seconds
    is_closed = close_ts <= now_ts
    qv_estimated = quote_volume is None
    if quote_volume is None:
        vwap = (high + low + close) / 3.0
        quote_volume = base_volume * vwap
    qc_flags = compute_qc_flags(open_, high, low, close, base_volume)

    doc = {
        "schema_version": SCHEMA_VERSION,
        "connector": connector,
        "trading_pair": hbot_pair,
        "interval": interval,
        "timestamp": ts_open,
        "open_ts": ts_open,
        "close_ts": close_ts,
        "is_closed": is_closed,
        "base_asset": base,
        "quote_asset": quote,
        "open": float(open_),
        "high": float(high),
        "low": float(low),
        "close": float(close),
        "volume": float(base_volume),
        "base_volume": float(base_volume),
        "quote_volume": float(quote_volume),
        "quote_volume_is_estimated": qv_estimated,
        "qc_ok": len(qc_flags) == 0,
        "qc_flags": qc_flags,
        "is_synthetic": False,
        "updated_at": now_ts,
    }
    if trade_count is not None: doc["trade_count"] = int(trade_count)
    if taker_buy_base_volume is not None: doc["taker_buy_base_volume"] = float(taker_buy_base_volume)
    if taker_buy_quote_volume is not None: doc["taker_buy_quote_volume"] = float(taker_buy_quote_volume)
    return doc


def upsert_candles(coll, candles, is_backfill=False):
    if not candles:
        return 0
    now_ts = utc_now_ts()
    if is_backfill:
        for c in candles:
            c.setdefault("ingested_at", now_ts)
        try:
            res = coll.insert_many(candles, ordered=False)
            return len(res.inserted_ids)
        except BulkWriteError as bwe:
            return int(bwe.details.get("nInserted", 0))
    ops = []
    for c in candles:
        key = {"connector": c["connector"], "trading_pair": c["trading_pair"],
               "interval": c["interval"], "timestamp": c["timestamp"]}
        ops.append(UpdateOne(key, {"$set": c, "$setOnInsert": {"ingested_at": now_ts}}, upsert=True))
    try:
        res = coll.bulk_write(ops, ordered=False)
        return int(res.upserted_count + res.modified_count)
    except BulkWriteError:
        return 0


print("HTTP + MongoDB helpers loaded.")

HTTP + MongoDB helpers loaded.


In [4]:
# ── Cell 4: Exchange-specific candle fetchers ────────────────────────────────

def fetch_nonkyc_candles(hbot_pair, interval, to_ts, count):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("nonkyc") is None:
        return []
    url = BASE_URLS["nonkyc"].rstrip("/") + "/market/candles"
    params = {
        "symbol": to_nonkyc_symbol(hbot_pair),
        "resolution": int(spec["nonkyc"]),
        "countBack": int(count),
        "to": int(to_ts),
    }
    data = http_get_json(url, params, prefix="[nonkyc] ")
    if not isinstance(data, dict):
        return []
    candles = []
    # Format 1: TradingView UDF arrays
    if data.get("s") == "ok" and "t" in data:
        t_arr = data.get("t", [])
        o_arr = data.get("o", [])
        h_arr = data.get("h", [])
        l_arr = data.get("l", [])
        c_arr = data.get("c", [])
        v_arr = data.get("v", [])
        for i in range(min(len(t_arr), len(o_arr), len(h_arr), len(l_arr), len(c_arr), len(v_arr))):
            candles.append({
                "timestamp": int(int(t_arr[i]) // 1000),
                "open": safe_float(o_arr[i]), "high": safe_float(h_arr[i]),
                "low": safe_float(l_arr[i]), "close": safe_float(c_arr[i]),
                "volume": safe_float(v_arr[i]),
            })
        return candles
    # Format 2: { bars: [...] }
    bars = data.get("bars")
    if isinstance(bars, list):
        for bar in bars:
            if not isinstance(bar, dict): continue
            ts_val = int(bar.get("time", 0))
            if ts_val > 1e12: ts_val = ts_val // 1000
            candles.append({
                "timestamp": ts_val,
                "open": safe_float(bar.get("open")), "high": safe_float(bar.get("high")),
                "low": safe_float(bar.get("low")), "close": safe_float(bar.get("close")),
                "volume": safe_float(bar.get("volume")),
            })
    return candles


def fetch_binance_candles(hbot_pair, interval, start_ms, end_ms, limit):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("binance") is None:
        return []
    url = BASE_URLS["binance"].rstrip("/") + "/api/v3/klines"
    params = {"symbol": to_binance_symbol(hbot_pair), "interval": spec["binance"],
              "startTime": int(start_ms), "endTime": int(end_ms), "limit": int(limit)}
    data = http_get_json(url, params, prefix="[binance] ")
    return data if isinstance(data, list) else []


def fetch_mexc_candles(hbot_pair, interval, start_ms, end_ms, limit):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("mexc") is None:
        return []
    url = BASE_URLS["mexc"].rstrip("/") + "/api/v3/klines"
    params = {"symbol": to_mexc_symbol(hbot_pair), "interval": spec["mexc"],
              "startTime": int(start_ms), "endTime": int(end_ms), "limit": int(limit)}
    data = http_get_json(url, params, prefix="[mexc] ")
    return data if isinstance(data, list) else []


def fetch_coinbase_candles(hbot_pair, interval, start_ts, end_ts):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("coinbase") is None:
        return []
    product_id = to_coinbase_product_id(hbot_pair)
    url = BASE_URLS["coinbase"].rstrip("/") + f"/products/{product_id}/candles"
    params = {"granularity": int(spec["coinbase"]),
              "start": ts_to_iso8601(int(start_ts)), "end": ts_to_iso8601(int(end_ts))}
    data = http_get_json(url, params, prefix="[coinbase] ")
    return data if isinstance(data, list) else []


print("Exchange fetchers loaded.")

Exchange fetchers loaded.


In [5]:
# ── Cell 5: Range ingesters (per exchange) ──────────────────────────────────

def ingest_nonkyc_range(coll, hbot_pair, interval, start_ts, end_ts):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("nonkyc") is None:
        return 0
    step = spec["seconds"]
    now_ts = utc_now_ts()
    max_per = MAX_PER_REQUEST["nonkyc"]
    delay = REQUEST_DELAYS["nonkyc"]
    page_to = end_ts + step - 1
    total = 0
    safety = 0
    while page_to >= start_ts and safety < 100000:
        safety += 1
        bars = fetch_nonkyc_candles(hbot_pair, interval, to_ts=page_to, count=max_per)
        if not bars:
            break
        docs = []
        oldest = None
        for b in bars:
            ts_open = int(b["timestamp"])
            if ts_open < start_ts or ts_open > end_ts:
                continue
            oldest = ts_open if oldest is None else min(oldest, ts_open)
            docs.append(normalize_candle_doc(
                connector="nonkyc", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=b["open"], high=b["high"], low=b["low"],
                close=b["close"], base_volume=b["volume"],
                interval_seconds=step, now_ts=now_ts,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        if oldest is None or oldest <= start_ts or oldest >= page_to:
            break
        page_to = oldest - 1
        time.sleep(delay)
    return total


def ingest_binance_range(coll, hbot_pair, interval, start_ts, end_ts):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("binance") is None:
        return 0
    step = spec["seconds"]
    now_ts = utc_now_ts()
    max_per = MAX_PER_REQUEST["binance"]
    delay = REQUEST_DELAYS["binance"]
    cur = start_ts
    total = 0
    safety = 0
    while cur <= end_ts and safety < 100000:
        safety += 1
        window_end = min(cur + (max_per - 1) * step, end_ts)
        rows = fetch_binance_candles(hbot_pair, interval, cur * 1000, window_end * 1000, max_per)
        if not rows:
            cur = window_end + step
            time.sleep(delay)
            continue
        docs = []
        max_row_ts = None
        for r in rows:
            ts_open = int(r[0]) // 1000
            if ts_open < start_ts or ts_open > end_ts:
                continue
            max_row_ts = ts_open if max_row_ts is None else max(max_row_ts, ts_open)
            docs.append(normalize_candle_doc(
                connector="binance", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=safe_float(r[1]), high=safe_float(r[2]),
                low=safe_float(r[3]), close=safe_float(r[4]),
                base_volume=safe_float(r[5]), interval_seconds=step, now_ts=now_ts,
                quote_volume=safe_float(r[7]),
                trade_count=int(r[8]) if len(r) > 8 else None,
                taker_buy_base_volume=safe_float(r[9]) if len(r) > 9 else None,
                taker_buy_quote_volume=safe_float(r[10]) if len(r) > 10 else None,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        cur = (max_row_ts + step) if max_row_ts else (cur + max_per * step)
        time.sleep(delay)
    return total


def ingest_mexc_range(coll, hbot_pair, interval, start_ts, end_ts):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("mexc") is None:
        return 0
    step = spec["seconds"]
    now_ts = utc_now_ts()
    max_per = MAX_PER_REQUEST["mexc"]
    delay = REQUEST_DELAYS["mexc"]
    cur = start_ts
    total = 0
    safety = 0
    while cur <= end_ts and safety < 100000:
        safety += 1
        window_end = min(cur + (max_per - 1) * step, end_ts)
        rows = fetch_mexc_candles(hbot_pair, interval, cur * 1000, window_end * 1000, max_per)
        if not rows:
            cur = window_end + step
            time.sleep(delay)
            continue
        docs = []
        max_row_ts = None
        for r in rows:
            ts_open = int(r[0]) // 1000
            if ts_open < start_ts or ts_open > end_ts:
                continue
            max_row_ts = ts_open if max_row_ts is None else max(max_row_ts, ts_open)
            docs.append(normalize_candle_doc(
                connector="mexc", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=safe_float(r[1]), high=safe_float(r[2]),
                low=safe_float(r[3]), close=safe_float(r[4]),
                base_volume=safe_float(r[5]), interval_seconds=step, now_ts=now_ts,
                quote_volume=safe_float(r[7]) if len(r) > 7 else None,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        cur = (max_row_ts + step) if max_row_ts else (cur + max_per * step)
        time.sleep(delay)
    return total


def ingest_coinbase_range(coll, hbot_pair, interval, start_ts, end_ts):
    spec = INTERVAL_SPEC.get(interval)
    if not spec or spec.get("coinbase") is None:
        return 0
    step = spec["seconds"]
    now_ts = utc_now_ts()
    max_per = MAX_PER_REQUEST["coinbase"]
    delay = REQUEST_DELAYS["coinbase"]
    cur = start_ts
    total = 0
    safety = 0
    while cur <= end_ts and safety < 100000:
        safety += 1
        window_end = min(cur + (max_per - 1) * step, end_ts)
        rows = fetch_coinbase_candles(hbot_pair, interval, cur, window_end)
        if not rows:
            cur = window_end + step
            time.sleep(delay)
            continue
        docs = []
        max_row_ts = None
        # Coinbase format: [time, low, high, open, close, volume] — time in seconds
        for r in rows:
            ts_open = int(r[0])
            if ts_open < start_ts or ts_open > end_ts:
                continue
            max_row_ts = ts_open if max_row_ts is None else max(max_row_ts, ts_open)
            docs.append(normalize_candle_doc(
                connector="coinbase", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=safe_float(r[3]), high=safe_float(r[2]),
                low=safe_float(r[1]), close=safe_float(r[4]),
                base_volume=safe_float(r[5]), interval_seconds=step, now_ts=now_ts,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        cur = (max_row_ts + step) if max_row_ts else (cur + max_per * step)
        time.sleep(delay)
    return total


INGESTERS = {
    "nonkyc": ingest_nonkyc_range,
    "binance": ingest_binance_range,
    "mexc": ingest_mexc_range,
    "coinbase": ingest_coinbase_range,
}

print("Range ingesters loaded.")

Range ingesters loaded.


In [6]:
# ── Cell 6: Connect to MongoDB & discover all combos ─────────────────────────

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
client.admin.command("ping")
db = client[MONGO_DATABASE]
coll = db[COLLECTION_NAME]

# Ensure indexes
coll.create_index(
    [("connector", 1), ("trading_pair", 1), ("interval", 1), ("timestamp", 1)],
    unique=True, name="idx_connector_pair_interval_ts",
)

print(f"Connected to MongoDB: {MONGO_URI.split('@')[-1]}")
print(f"Database: {MONGO_DATABASE}, Collection: {COLLECTION_NAME}")
print(f"Total documents: {coll.estimated_document_count():,}")

Connected to MongoDB: 192.168.1.54:27017/quants_lab?authSource=admin&retryWrites=true&w=majority
Database: quants_lab, Collection: candles
Total documents: 7,176,702


In [7]:
# ── Cell 7: Scan all combos and compute gap statistics ───────────────────────

now_ts = utc_now_ts()
window_start = now_ts - BACKFILL_DAYS * 86400

# Aggregation: discover all (connector, pair, interval) combos
pipeline = [
    {"$group": {
        "_id": {"connector": "$connector", "trading_pair": "$trading_pair", "interval": "$interval"},
        "count": {"$sum": 1},
        "min_ts": {"$min": "$timestamp"},
        "max_ts": {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.trading_pair": 1, "_id.interval": 1}},
]

raw_combos = list(coll.aggregate(pipeline, allowDiskUse=True))

combos = []
for doc in raw_combos:
    connector = doc["_id"]["connector"]
    pair = doc["_id"]["trading_pair"]
    interval = doc["_id"]["interval"]
    spec = INTERVAL_SPEC.get(interval)
    if not spec:
        continue
    step = spec["seconds"]
    min_ts = max(int(doc["min_ts"]), window_start)
    max_ts = int(doc["max_ts"])
    
    # Count candles in our 180-day window
    candles_in_window = coll.count_documents({
        "connector": connector, "trading_pair": pair, "interval": interval,
        "timestamp": {"$gte": window_start}
    })
    
    # Expected candles in the 180-day window (up to last closed candle)
    effective_start = align_floor(window_start, step)
    effective_end = last_closed_open_ts(now_ts, step)
    expected = ((effective_end - effective_start) // step) + 1 if effective_end >= effective_start else 0
    missing = max(0, expected - candles_in_window)
    pct_complete = (candles_in_window / expected * 100) if expected > 0 else 100.0
    
    combos.append({
        "connector": connector,
        "pair": pair,
        "interval": interval,
        "total_docs": int(doc["count"]),
        "in_window": candles_in_window,
        "expected": expected,
        "missing": missing,
        "pct": pct_complete,
        "step": step,
    })

# Group by connector for display
connectors_found = sorted(set(c["connector"] for c in combos))

print(f"Found {len(combos)} (connector, pair, interval) combos across {len(connectors_found)} connector(s)")
print(f"180-day window: {fmt_ts(window_start)} → {fmt_ts(now_ts)}")

# ── Connector summary ──
print(f"\nConnector Summary:")
for cidx, conn in enumerate(connectors_found):
    conn_combos = [c for c in combos if c["connector"] == conn]
    total_missing = sum(c["missing"] for c in conn_combos)
    pairs_count = len(set(c["pair"] for c in conn_combos))
    intervals_count = len(set(c["interval"] for c in conn_combos))
    row_indices = [i for i, c in enumerate(combos) if c["connector"] == conn]
    print(f"  {conn:12s} — {pairs_count} pairs × {intervals_count} intervals = {len(conn_combos)} series, "
          f"{total_missing:,} missing  (rows {row_indices[0]}–{row_indices[-1]})")

# ── Detail table with row index ──
print(f"\n{'─' * 100}")
print(f"{'Idx':>5s}  {'Connector':12s} {'Pair':16s} {'Intv':6s} {'In Window':>10s} {'Expected':>10s} {'Missing':>10s} {'Complete':>9s}")
print(f"{'─' * 100}")
for i, c in enumerate(combos):
    flag = " ✓" if c["missing"] == 0 else f" ← {c['missing']:,}"
    print(f"{i:>5d}  {c['connector']:12s} {c['pair']:16s} {c['interval']:6s} "
          f"{c['in_window']:>10,d} {c['expected']:>10,d} {c['missing']:>10,d} {c['pct']:>8.1f}%{flag}")
print(f"{'─' * 100}")
print(f"\nUse these indices in Cell 8 to select what to backfill.")


Found 326 (connector, pair, interval) combos across 3 connector(s)
180-day window: 2025-09-28 19:22 → 2026-03-27 19:22

Connector Summary:
  coinbase     — 3 pairs × 5 intervals = 15 series, 729 missing  (rows 0–14)
  mexc         — 31 pairs × 6 intervals = 186 series, 6,434,203 missing  (rows 15–200)
  nonkyc       — 25 pairs × 5 intervals = 125 series, 84,765 missing  (rows 201–325)

────────────────────────────────────────────────────────────────────────────────────────────────────
  Idx  Connector    Pair             Intv    In Window   Expected    Missing  Complete
────────────────────────────────────────────────────────────────────────────────────────────────────
    0  coinbase     BTC-USD          15m        17,267     17,280         13     99.9% ← 13
    1  coinbase     BTC-USD          1d            179        180          1     99.4% ← 1
    2  coinbase     BTC-USD          1h          4,316      4,320          4     99.9% ← 4
    3  coinbase     BTC-USD          1m        2

In [8]:
# ── Cell 8: Select what to backfill ──────────────────────────────────────────
#
# Choose ONE of these selection modes:
#
#   SELECTION = "all"              # backfill every combo with gaps
#   SELECTION = "nonkyc"           # backfill all series for a connector name
#   SELECTION = "mexc"             # backfill all series for a connector name
#   SELECTION = [0, 1, 2]          # backfill specific row indices from the table above
#   SELECTION = range(0, 25)       # backfill a range of row indices
#   SELECTION = [5]                # backfill a single row

SELECTION = "all"   # ← CHANGE THIS

# ── Resolve selection to a list of combos ──
if SELECTION == "all":
    selected_combos = [c for c in combos if c["missing"] > 0]
    sel_desc = "all connectors"
elif isinstance(SELECTION, str):
    # Connector name
    selected_combos = [c for c in combos if c["connector"] == SELECTION and c["missing"] > 0]
    sel_desc = f"connector '{SELECTION}'"
else:
    # List or range of row indices
    indices = list(SELECTION)
    selected_combos = [combos[i] for i in indices if i < len(combos) and combos[i]["missing"] > 0]
    sel_desc = f"rows {indices}"

total_to_fill = sum(c["missing"] for c in selected_combos)
print(f"Selection: {sel_desc}")
print(f"Series with gaps: {len(selected_combos)}")
print(f"Total missing candles to fill: {total_to_fill:,}")

if selected_combos:
    print(f"\nWill backfill:")
    for c in selected_combos:
        print(f"  {c['connector']:12s} {c['pair']:16s} {c['interval']:6s} — {c['missing']:,} missing")
    print(f"\nReady. Run the next cell to start.")
else:
    print("\n✓ Nothing to backfill — all selected series are complete!")


Selection: all connectors
Series with gaps: 326
Total missing candles to fill: 6,519,697

Will backfill:
  coinbase     BTC-USD          15m    — 13 missing
  coinbase     BTC-USD          1d     — 1 missing
  coinbase     BTC-USD          1h     — 4 missing
  coinbase     BTC-USD          1m     — 187 missing
  coinbase     BTC-USD          5m     — 38 missing
  coinbase     ETH-USD          15m    — 13 missing
  coinbase     ETH-USD          1d     — 1 missing
  coinbase     ETH-USD          1h     — 4 missing
  coinbase     ETH-USD          1m     — 187 missing
  coinbase     ETH-USD          5m     — 38 missing
  coinbase     USDT-USD         15m    — 13 missing
  coinbase     USDT-USD         1d     — 1 missing
  coinbase     USDT-USD         1h     — 4 missing
  coinbase     USDT-USD         1m     — 187 missing
  coinbase     USDT-USD         5m     — 38 missing
  mexc         ADA-USDT         15m    — 14 missing
  mexc         ADA-USDT         1d     — 2 missing
  mexc         

In [9]:
# ── Cell 9: Gap detection (streaming, memory-safe) ──────────────────────────

def find_gaps_in_window(coll, connector, pair, interval, step, window_start, window_end):
    """
    Find all gap ranges within the specified window.
    Also detects leading gaps (window_start to first candle) and trailing gaps
    (last candle to window_end).
    Returns list of (gap_start_ts, gap_end_ts) tuples.
    """
    effective_start = align_floor(window_start, step)
    effective_end = last_closed_open_ts(utc_now_ts(), step)
    
    if effective_end < effective_start:
        return []
    
    # Stream timestamps in ascending order
    cursor = coll.find(
        {"connector": connector, "trading_pair": pair, "interval": interval,
         "timestamp": {"$gte": effective_start, "$lte": effective_end}},
        projection={"timestamp": 1, "_id": 0},
        sort=[("timestamp", 1)],
        batch_size=5000,
    )
    
    ranges = []
    prev = None
    first_ts = None
    last_ts = None
    
    for doc in cursor:
        ts = int(doc["timestamp"])
        if first_ts is None:
            first_ts = ts
        last_ts = ts
        
        if prev is not None:
            expected_next = prev + step
            if ts > expected_next:
                ranges.append((expected_next, ts - step))
        prev = ts
    
    # Leading gap: window start to first candle
    if first_ts is not None and first_ts > effective_start:
        ranges.insert(0, (effective_start, first_ts - step))
    elif first_ts is None:
        # No candles at all in the window — entire range is a gap
        ranges.append((effective_start, effective_end))
    
    # Trailing gap: last candle to window end
    if last_ts is not None and last_ts < effective_end:
        trailing_start = last_ts + step
        if trailing_start <= effective_end:
            ranges.append((trailing_start, effective_end))
    
    # Filter out any invalid ranges
    ranges = [(s, e) for s, e in ranges if s <= e]
    
    return ranges


print("Gap detection loaded.")

Gap detection loaded.


In [10]:
# ── Cell 10: Run the backfill ────────────────────────────────────────────────
#
# This cell iterates through every (connector, pair, interval) combo that has
# gaps, finds the exact missing ranges, and fetches them from the exchange API.

if not selected_combos:
    print("Nothing to backfill.")
else:
    grand_total = 0
    grand_gaps = 0
    errors = []
    
    for i, combo in enumerate(selected_combos):
        connector = combo["connector"]
        pair = combo["pair"]
        interval = combo["interval"]
        step = combo["step"]
        
        ingester = INGESTERS.get(connector)
        if ingester is None:
            print(f"  ⚠ No ingester for '{connector}' — skipping")
            continue
        
        # Check that this connector supports this interval
        spec = INTERVAL_SPEC.get(interval)
        if not spec or spec.get(connector) is None:
            print(f"  ⚠ {connector} does not support {interval} — skipping {pair}")
            continue
        
        print(f"\n[{i+1}/{len(selected_combos)}] {connector} {pair} {interval} (est. {combo['missing']:,} missing)")
        
        # Find exact gaps
        gaps = find_gaps_in_window(coll, connector, pair, interval, step, window_start, now_ts)
        
        if not gaps:
            print(f"  ✓ No gaps found (may have been filled already)")
            continue
        
        total_missing_candles = sum(((e - s) // step) + 1 for s, e in gaps)
        print(f"  Found {len(gaps)} gap range(s), {total_missing_candles:,} missing candles")
        grand_gaps += len(gaps)
        
        combo_written = 0
        for gap_idx, (gs, ge) in enumerate(gaps):
            candles_needed = ((ge - gs) // step) + 1
            if gap_idx < 3 or gap_idx == len(gaps) - 1:  # Print first 3 and last
                print(f"    Range {gap_idx+1}/{len(gaps)}: {fmt_ts(gs)} → {fmt_ts(ge)} ({candles_needed:,} candles)")
            elif gap_idx == 3:
                print(f"    ... ({len(gaps) - 4} more ranges) ...")
            
            try:
                written = ingester(coll, pair, interval, gs, ge)
                combo_written += written
            except Exception as e:
                err_msg = f"{connector} {pair} {interval} range {fmt_ts(gs)}→{fmt_ts(ge)}: {e}"
                errors.append(err_msg)
                print(f"    ✗ ERROR: {e}")
        
        grand_total += combo_written
        print(f"  → Wrote {combo_written:,} candles")
    
    print(f"\n{'═' * 70}")
    print(f"BACKFILL COMPLETE")
    print(f"  Gap ranges processed: {grand_gaps:,}")
    print(f"  Total candles written: {grand_total:,}")
    if errors:
        print(f"  Errors: {len(errors)}")
        for e in errors[:10]:
            print(f"    ✗ {e}")
        if len(errors) > 10:
            print(f"    ... and {len(errors) - 10} more")
    else:
        print(f"  Errors: 0 ✓")
    print(f"{'═' * 70}")


[1/326] coinbase BTC-USD 15m (est. 13 missing)
  Found 1 gap range(s), 12 missing candles
    Range 1/1: 2026-03-27 16:15 → 2026-03-27 19:00 (12 candles)
  → Wrote 12 candles

[2/326] coinbase BTC-USD 1d (est. 1 missing)
  ✓ No gaps found (may have been filled already)

[3/326] coinbase BTC-USD 1h (est. 4 missing)
  Found 1 gap range(s), 3 missing candles
    Range 1/1: 2026-03-27 16:00 → 2026-03-27 18:00 (3 candles)
  → Wrote 3 candles

[4/326] coinbase BTC-USD 1m (est. 187 missing)
  Found 1 gap range(s), 186 missing candles
    Range 1/1: 2026-03-27 16:16 → 2026-03-27 19:21 (186 candles)
  → Wrote 186 candles

[5/326] coinbase BTC-USD 5m (est. 38 missing)
  Found 1 gap range(s), 37 missing candles
    Range 1/1: 2026-03-27 16:15 → 2026-03-27 19:15 (37 candles)
  → Wrote 37 candles

[6/326] coinbase ETH-USD 15m (est. 13 missing)
  Found 1 gap range(s), 12 missing candles
    Range 1/1: 2026-03-27 16:15 → 2026-03-27 19:00 (12 candles)
  → Wrote 12 candles

[7/326] coinbase ETH-USD 1d

In [11]:
# ── Cell 11: Post-backfill verification ──────────────────────────────────────
#
# Re-scan the selected combos to verify completeness after backfill.

# Derive the set of connectors from what was selected in Cell 8
selected_connectors_set = set(c["connector"] for c in selected_combos)

print(f"Post-backfill verification ({', '.join(sorted(selected_connectors_set))})")
print(f"{'─' * 100}")
print(f"{'Idx':>5s}  {'Connector':12s} {'Pair':16s} {'Intv':6s} {'In Window':>10s} {'Expected':>10s} {'Missing':>10s} {'Complete':>9s}")
print(f"{'─' * 100}")

still_missing_total = 0

for i, c in enumerate(combos):
    if c["connector"] not in selected_connectors_set:
        continue
    
    # Recount
    in_window = coll.count_documents({
        "connector": c["connector"], "trading_pair": c["pair"], "interval": c["interval"],
        "timestamp": {"$gte": window_start}
    })
    missing = max(0, c["expected"] - in_window)
    pct = (in_window / c["expected"] * 100) if c["expected"] > 0 else 100.0
    still_missing_total += missing
    flag = " ✓" if missing == 0 else f" ← {missing:,} still missing"
    print(f"{i:>5d}  {c['connector']:12s} {c['pair']:16s} {c['interval']:6s} "
          f"{in_window:>10,d} {c['expected']:>10,d} {missing:>10,d} {pct:>8.1f}%{flag}")

print(f"{'─' * 100}")
if still_missing_total == 0:
    print("✓ All series are 100% complete!")
else:
    print(f"⚠ {still_missing_total:,} candles still missing (exchange may not have data for those periods)")


Post-backfill verification (coinbase, mexc, nonkyc)
────────────────────────────────────────────────────────────────────────────────────────────────────
  Idx  Connector    Pair             Intv    In Window   Expected    Missing  Complete
────────────────────────────────────────────────────────────────────────────────────────────────────
    0  coinbase     BTC-USD          15m        17,279     17,280          1    100.0% ← 1 still missing
    1  coinbase     BTC-USD          1d            179        180          1     99.4% ← 1 still missing
    2  coinbase     BTC-USD          1h          4,319      4,320          1    100.0% ← 1 still missing
    3  coinbase     BTC-USD          1m        259,199    259,200          1    100.0% ← 1 still missing
    4  coinbase     BTC-USD          5m         51,839     51,840          1    100.0% ← 1 still missing
    5  coinbase     ETH-USD          15m        17,279     17,280          1    100.0% ← 1 still missing
    6  coinbase     ETH-USD  

In [12]:
# ── Cell 12 (Optional): Inspect a specific series ───────────────────────────
#
# Change these to drill into a specific pair's gap details.

INSPECT_CONNECTOR = "nonkyc"    # ← change as needed
INSPECT_PAIR = "BTC-USDT"       # ← change as needed
INSPECT_INTERVAL = "5m"         # ← change as needed

spec = INTERVAL_SPEC.get(INSPECT_INTERVAL)
if spec:
    step = spec["seconds"]
    gaps = find_gaps_in_window(coll, INSPECT_CONNECTOR, INSPECT_PAIR, INSPECT_INTERVAL, step, window_start, now_ts)
    
    total_missing = sum(((e - s) // step) + 1 for s, e in gaps)
    print(f"Gaps for {INSPECT_CONNECTOR} {INSPECT_PAIR} {INSPECT_INTERVAL}:")
    print(f"  Total gap ranges: {len(gaps)}")
    print(f"  Total missing candles: {total_missing:,}")
    print()
    for i, (gs, ge) in enumerate(gaps[:20]):
        cnt = ((ge - gs) // step) + 1
        dur_hours = cnt * step / 3600
        print(f"  [{i+1:3d}] {fmt_ts(gs)} → {fmt_ts(ge)}  ({cnt:>6,d} candles, {dur_hours:>7.1f}h)")
    if len(gaps) > 20:
        print(f"  ... and {len(gaps) - 20} more ranges")
else:
    print(f"Unknown interval: {INSPECT_INTERVAL}")

Gaps for nonkyc BTC-USDT 5m:
  Total gap ranges: 0
  Total missing candles: 0

